<a href="https://colab.research.google.com/github/ajdanoff/mcp-robot-welding-sim/blob/main/notebooks/cuda_morton_broadphase_benchmark.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
%%writefile morton_gpu.cu
#include <cuda_runtime.h>
#include <algorithm>
#include <chrono>
#include <cmath>
#include <cstdint>
#include <cstdio>
#include <vector>

__device__ __forceinline__ uint32_t expand_bits_gpu(uint32_t v) {
    v = (v * 0x00010001u) & 0xFF0000FFu;
    v = (v * 0x00000101u) & 0x0F00F00Fu;
    v = (v * 0x00000011u) & 0xC30C30C3u;
    v = (v * 0x00000005u) & 0x49249249u;
    return v;
}

__global__ void compute_morton_kernel(
    const float* __restrict__ xs,
    const float* __restrict__ ys,
    const float* __restrict__ zs,
    uint32_t* __restrict__ out_codes,
    uint32_t count
) {
    const uint32_t idx = blockIdx.x * blockDim.x + threadIdx.x;
    const uint32_t stride = blockDim.x * gridDim.x;

    for (uint32_t i = idx; i < count; i += stride) {
        const float x = fminf(fmaxf(xs[i] * 1024.0f, 0.0f), 1023.0f);
        const float y = fminf(fmaxf(ys[i] * 1024.0f, 0.0f), 1023.0f);
        const float z = fminf(fmaxf(zs[i] * 1024.0f, 0.0f), 1023.0f);

        const uint32_t xx = expand_bits_gpu(static_cast<uint32_t>(x));
        const uint32_t yy = expand_bits_gpu(static_cast<uint32_t>(y));
        const uint32_t zz = expand_bits_gpu(static_cast<uint32_t>(z));

        out_codes[i] = (xx << 2) | (yy << 1) | zz;
    }
}

extern "C" float benchmark_morton_gpu(
    const float* h_xs,
    const float* h_ys,
    const float* h_zs,
    uint32_t* h_codes,
    uint32_t count
) {
    const size_t float_bytes = count * sizeof(float);
    const size_t code_bytes = count * sizeof(uint32_t);

    float *d_xs = nullptr, *d_ys = nullptr, *d_zs = nullptr;
    uint32_t *d_codes = nullptr;

    cudaMalloc(&d_xs, float_bytes);
    cudaMalloc(&d_ys, float_bytes);
    cudaMalloc(&d_zs, float_bytes);
    cudaMalloc(&d_codes, code_bytes);

    cudaMemcpy(d_xs, h_xs, float_bytes, cudaMemcpyHostToDevice);
    cudaMemcpy(d_ys, h_ys, float_bytes, cudaMemcpyHostToDevice);
    cudaMemcpy(d_zs, h_zs, float_bytes, cudaMemcpyHostToDevice);

    cudaEvent_t start, stop;
    cudaEventCreate(&start);
    cudaEventCreate(&stop);

    const int threads = 256;
    const int blocks = std::min<int>((count + threads - 1) / threads, 1024);

    cudaEventRecord(start);
    compute_morton_kernel<<<blocks, threads>>>(d_xs, d_ys, d_zs, d_codes, count);
    cudaEventRecord(stop);
    cudaEventSynchronize(stop);

    float kernel_ms = 0.0f;
    cudaEventElapsedTime(&kernel_ms, start, stop);

    cudaMemcpy(h_codes, d_codes, code_bytes, cudaMemcpyDeviceToHost);

    cudaEventDestroy(start);
    cudaEventDestroy(stop);
    cudaFree(d_xs);
    cudaFree(d_ys);
    cudaFree(d_zs);
    cudaFree(d_codes);

    return kernel_ms;
}


Writing morton_gpu.cu


In [2]:
!nvidia-smi --query-gpu=name,compute_cap,memory.total --format=csv
!nvcc -O3 -arch=sm_75 -shared -Xcompiler -fPIC morton_gpu.cu -o libmorton_gpu.so

name, compute_cap, memory.total [MiB]
Tesla T4, 7.5, 15360 MiB


In [3]:
import ctypes
import time
import numpy as np

# Load compiled CUDA shared library
lib = ctypes.CDLL("./libmorton_gpu.so")
lib.benchmark_morton_gpu.argtypes = [
    np.ctypeslib.ndpointer(dtype=np.float32, flags="C_CONTIGUOUS"),
    np.ctypeslib.ndpointer(dtype=np.float32, flags="C_CONTIGUOUS"),
    np.ctypeslib.ndpointer(dtype=np.float32, flags="C_CONTIGUOUS"),
    np.ctypeslib.ndpointer(dtype=np.uint32, flags="C_CONTIGUOUS"),
    ctypes.c_uint32,
]
lib.benchmark_morton_gpu.restype = ctypes.c_float

# Generate 1,000,000 normalized 3D CAD mesh primitives (SoA layout for coalesced GPU reads)
N = 1_000_000
rng = np.random.default_rng(42)
xs = rng.random(N, dtype=np.float32)
ys = rng.random(N, dtype=np.float32)
zs = rng.random(N, dtype=np.float32)
gpu_codes = np.zeros(N, dtype=np.uint32)

# 1. GPU CUDA Kernel Execution
gpu_kernel_ms = lib.benchmark_morton_gpu(xs, ys, zs, gpu_codes, N)

# 2. CPU NumPy Vectorized Reference
def morton_cpu_numpy(x: np.ndarray, y: np.ndarray, z: np.ndarray) -> np.ndarray:
    def expand(v: np.ndarray) -> np.ndarray:
        v = (v * 0x00010001) & 0xFF0000FF
        v = (v * 0x00000101) & 0x0F00F00F
        v = (v * 0x00000011) & 0xC30C30C3
        v = (v * 0x00000005) & 0x49249249
        return v

    ix = np.clip((x * 1024.0).astype(np.uint32), 0, 1023)
    iy = np.clip((y * 1024.0).astype(np.uint32), 0, 1023)
    iz = np.clip((z * 1024.0).astype(np.uint32), 0, 1023)
    return (expand(ix) << 2) | (expand(iy) << 1) | expand(iz)

t0 = time.perf_counter()
cpu_codes = morton_cpu_numpy(xs, ys, zs)
cpu_ms = (time.perf_counter() - t0) * 1000.0

assert np.array_equal(gpu_codes, cpu_codes), "Mismatch between GPU and CPU Morton codes!"

print(f"=== CUDA Spatial Indexing Benchmark ({N:,} 3D Primitives) ===")
print(f"Bit-exact verification : PASSED (100% match)")
print(f"CPU NumPy Time         : {cpu_ms:.2f} ms")
print(f"GPU CUDA Kernel Time   : {gpu_kernel_ms:.3f} ms")
print(f"Kernel Speedup         : {cpu_ms / gpu_kernel_ms:.1f}x faster on NVIDIA T4")


=== CUDA Spatial Indexing Benchmark (1,000,000 3D Primitives) ===
Bit-exact verification : PASSED (100% match)
CPU NumPy Time         : 37.23 ms
GPU CUDA Kernel Time   : 0.201 ms
Kernel Speedup         : 185.2x faster on NVIDIA T4
